In [1]:
import os
import json
from tqdm import tqdm

In [2]:
def read_jsonl_file(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            json_data = json.loads(line)
            data.append(json_data['canonical_smiles'])
    return data

In [3]:
chembl_filepath = os.path.join("../../data","parsed-data","chembl_36_data.jsonl")
chembl_smiles = read_jsonl_file(chembl_filepath)

In [4]:
print(f"Number of SMILES in ChEMLB as of 2025-11-16: {len(chembl_smiles)}")

Number of SMILES in ChEMLB as of 2025-11-16: 2854815


In [5]:
chembl_smiles[0]

'Cc1cc(-c2csc(N=C(N)N)n2)cn1C'

In [6]:
def get_ngrams(smiles_list, ngram_size=1):
    dict_ngrams = {}
    for smiles in tqdm(chembl_smiles):
        smiles_split = list(smiles)
        for char_range in range(0, len(smiles_split)-ngram_size+1):
            ngram = ''.join(smiles_split[char_range:char_range+ngram_size])
            if ngram not in dict_ngrams:
                dict_ngrams[ngram] = 0
            dict_ngrams[ngram] += 1
    return dict_ngrams

In [7]:
data_unigram = get_ngrams(chembl_smiles, ngram_size=1)

100%|██████████| 2854815/2854815 [00:18<00:00, 158486.76it/s]


In [8]:
print(f"Number of unique characters in ChEMBL SMILES data: {len(data_unigram)}")

Number of unique characters in ChEMBL SMILES data: 55


In [9]:
print(f"Sorted chars from SMILES: {sorted(data_unigram.keys())}")

Sorted chars from SMILES: ['#', '%', '(', ')', '+', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '@', 'A', 'B', 'C', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'X', 'Y', 'Z', '[', '\\', ']', 'a', 'b', 'c', 'e', 'g', 'i', 'l', 'n', 'o', 'p', 'r', 's', 't']


In [10]:
print(data_unigram)

{'C': 30153372, 'c': 35305028, '1': 8540736, '(': 16053159, '-': 1221303, '2': 6888130, 's': 342368, 'N': 6167774, '=': 6258998, ')': 16053159, 'n': 4488020, '[': 3941990, '@': 4615977, 'H': 3205247, ']': 3941990, 'O': 9629098, 'S': 751974, '3': 3770494, 'l': 710205, 'B': 144746, 'r': 137480, '/': 578512, '\\': 131373, 'o': 321626, '4': 1396112, '5': 393888, '6': 95519, '7': 25902, '+': 200369, '.': 148935, 'I': 22681, 'F': 1467276, '8': 10479, '#': 216008, 'P': 60610, '9': 4083, 'a': 16555, '%': 8416, '0': 2499, 'i': 4911, 'e': 4310, 'L': 577, 'K': 1625, 't': 74, 'T': 154, 'A': 483, 'g': 174, 'Z': 165, 'M': 132, 'R': 16, 'p': 37, 'b': 23, 'Y': 1, 'G': 1, 'X': 4}


In [11]:
data_bigram = get_ngrams(chembl_smiles, ngram_size=2)

100%|██████████| 2854815/2854815 [00:24<00:00, 115295.42it/s]


In [12]:
print(f"Number of bigram in ChEMBL SMILES data: {len(data_bigram)}")

Number of bigram in ChEMBL SMILES data: 864


# SMILES Introduction
- We cannot lowercase the input SMILES as the non-caps 'c' denotes aromatic while, caps 'C' denotes aliphatic!

# N-gram
- It doesn't make sense to have 'B' and 'r' as separate when SMILES meant 'Br', which in chemistry is an element called Bromine (Br).
- As we increase the N-gram size the time complexity will also increase.

In [17]:
class Tokenizer:
    def __init__(self, tokenizer_type:str="ngram", ngram_size:int=0) -> None:
        self.vocab = {'<UNK>':0}
        self.inv_vocab = {0:'<UNK>'}
        self.tokenizer_type = tokenizer_type
        self.ngram_size = ngram_size
        #
        self._initialization_checks

    def _initialization_checks(self):
        if self.tokenizer_type=="ngram":
            if self.ngram_size==0:
                print(f"N-gram size cannot be set to 0. You need to provide size as parameter ngram_size with value greater than 1.")
    
    def _ngrams(self, smiles_list):
        dict_ngrams = {}
        for smiles in tqdm(chembl_smiles):
            smiles_split = list(smiles)
            for char_range in range(0, len(smiles_split)-self.ngram_size+1):
                ngram = ''.join(smiles_split[char_range:char_range+self.ngram_size])
                if ngram not in dict_ngrams:
                    dict_ngrams[ngram] = 0
                dict_ngrams[ngram] += 1
        return sorted(data_unigram.keys())
    
    def build_vocab(self, smiles_list:list=[]):
        # Build vocab
        if self.tokenizer_type=="ngram":
            sorted_ngrams = self._ngrams(smiles_list=smiles_list)
            #
            for counter,ngram in enumerate(sorted_ngrams):
                self.vocab[ngram] = counter+1
                self.inv_vocab[counter+1] = ngram
    
    def tokenize(self, smiles:str):
        if self.tokenizer_type=="ngram":
            tokens = []
            smiles_split = list(smiles)
            for char_range in range(0, len(smiles_split)-self.ngram_size+1):
                ngram = ''.join(smiles_split[char_range:char_range+self.ngram_size])
                tokens.append(self.vocab.get(ngram, 0))
            return tokens
        
    def detokenize(self, token_ids:list=[]):
        if self.tokenizer_type=="ngram":
            ngrams = []
            for token_id in token_ids:
                ngram = self.inv_vocab.get(token_id, '<UNK>')
                ngrams.append(ngram)
            return ''.join(ngrams)


In [19]:
obj_tokenizer = Tokenizer(tokenizer_type="ngram", ngram_size=1)
obj_tokenizer.build_vocab(smiles_list=chembl_smiles)

100%|██████████| 2854815/2854815 [00:18<00:00, 151748.77it/s]


In [20]:
obj_tokenizer.tokenize(smiles="CCO")

[23, 23, 32]

In [21]:
obj_tokenizer.detokenize(token_ids=[23, 23, 32])

'CCO'